```
<08-2_data_renewal.ipynb>

제미나이 의존도: 30-40%

08_data_renewal.ipynb와 차이점은, 직접 만든 합성곱 신경망 대신, 백본 모델(ResNet18)을 개조해서 사용했다는 점이다.
08_data_renewal.ipynb때도 정확도가 높았지만, 바나나를 썩은 사과로 예측하는 걸 보고
이미지의 전체적인 윤곽을 보지 못하고 있구나 라고 생각해서 모델을 바꾸게 되었다.

Epoch 1/5 | Train Loss: 0.1760
Epoch 2/5 | Train Loss: 0.0678
Epoch 3/5 | Train Loss: 0.0305
Epoch 4/5 | Train Loss: 0.0585
Epoch 5/5 | Train Loss: 0.0352
Test Accuracy: 99.61%

모델 하나 바뀌었을 뿐인데 정확도가 97%에서 99%가 되었다.
전 모델이 틀린 문제를 이 모델에게 주었을 때는 대체로 잘 맞히는 모습을 볼 수 있었다.

한가지 아쉬운 점은, lr이 현재 손실 지형에 비해 조금 커서 바닥(최저점)을 밟지 못하고 Overshooting을 한 것으로 보인다.
이것을 개선하기 위해서는 lr scheduler 사용을 해야 한다.
학습 초반에는 빠르게 Loss를 줄이고, 후반에는 lr을 조금 낮춰서 바닥(최저점)에 안정적으로 안착하게 하는 게 목표다.
```

In [9]:
import os
from sklearn.model_selection import train_test_split
import pandas as pd

fruits = ["apple", "banana", "orange"]

classes = []

data = []

for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]

    classes.extend(categories)
    
    for category in categories:
        folder_path = os.path.join(path, category)
        for image in os.listdir(folder_path):
            if image.endswith(('jpg', 'jpeg', 'png')):
                full_path = os.path.join(folder_path, image)
                data.append({
                    "image_path": full_path, 
                    "label_name": category
                })

df = pd.DataFrame(data)
print(len(df)) # 16831

print(classes)
# ['fresh_apple', 'rotten_apple', 'fresh_banana', 'rotten_banana', 'fresh_orange', 'rotten_orange']

class_map = {name: i for i, name in enumerate(classes)}
df['label'] = df['label_name'].map(class_map)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df.reset_index(drop=True)
test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

16831
['fresh_apple', 'rotten_apple', 'fresh_banana', 'rotten_banana', 'fresh_orange', 'rotten_orange']
Train: 13464, Test: 3367


In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2

class FruitsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = row['image_path']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']
        
        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long)

In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(128, 128), 
    A.HorizontalFlip(p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

train_dataset = FruitsDataset(df=train_df, transform=train_transform)
test_dataset = FruitsDataset(df=test_df, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [13]:
images, labels = next(iter(train_loader))
print(images.shape)

torch.Size([64, 3, 128, 128])


In [14]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# ImageNet으로 사전학습된 가중치를 가진 ResNet18 불러오기
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 기존 fc 레이어의 입력 특징 수(in_features)가 몇개인지 알아내기
num_features = model.fc.in_features

# 모델의 fc 자리에 우리의 분류 개수(6)에 맞게 개조
model.fc = nn.Linear(num_features, 6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/jeongjaehun/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:12<00:00, 3.78MB/s]


In [16]:
epochs = 5

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader.dataset):.4f}")

model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()

print(f"Test Accuracy: {correct/len(test_loader.dataset) * 100:.2f}%")

Epoch 1/5 | Train Loss: 0.1760
Epoch 2/5 | Train Loss: 0.0678
Epoch 3/5 | Train Loss: 0.0305
Epoch 4/5 | Train Loss: 0.0585
Epoch 5/5 | Train Loss: 0.0352
Test Accuracy: 99.61%


In [17]:
torch.save(model.state_dict(), 'resnet18_fruits.pth')